# 🏢 Module 7 — Lab: Enterprise Multi-Agent Department

**Mục tiêu:** Xây dựng toàn bộ LoanBot v3.0 — 8-agent enterprise department với:
- ✅ Agent Registry & ANS (Agent Name Service)
- ✅ Capability Cards với TTL lease
- ✅ A2A Protocol Task State Machine
- ✅ State Checkpointing & Resume
- ✅ Cascading Failure Protection (Circuit Breaker)
- ✅ 8 Specialized Agents trên Claude API thật

**Case study:** FinTech Corp — LoanBot v3.0 xử lý 50,000 hồ sơ vay/tháng

---
```
Thời gian ước tính: 45-60 phút
API: Claude claude-haiku-4-5-20251001 (tiết kiệm chi phí)
Packages: anthropic, asyncio (built-in), json, datetime
```

## ⚙️ Setup — Cài đặt môi trường

In [ ]:
!pip install anthropic --quiet

import os
import json
import time
import asyncio
import hashlib
import anthropic
from datetime import datetime, timedelta
from enum import Enum
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Any

# ── API Key
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-api03-YOUR_KEY_HERE"  # Điền API key của bạn
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"

print("✅ Setup hoàn tất!")
print(f"🤖 Model: {MODEL}")
print(f"📅 Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 🔷 Phần 1 — Agent Registry & ANS Pattern

**Vấn đề:** LoanBot v2.0 hardcoded tên agent → fragile khi scale

**Giải pháp:** Agent Registry — DNS cho agents. Supervisor query theo **capability**, không theo **tên**.

```
Trước (v2.0):  supervisor.agents['credit_agent'].run()
Sau (v3.0):    registry.find(capability='credit_check')[0].run()
```

In [ ]:
# ─── Bước 1.1: Định nghĩa CapabilityCard
# "CV" của mỗi agent — registry dùng để discovery và routing

@dataclass
class CapabilityCard:
    agent_id: str
    name: str
    capabilities: List[str]          # ['credit_check', 'cic_query']
    sla_ms: int                       # Cam kết latency tối đa
    cost_per_call: float              # USD
    input_schema: Dict[str, str]
    output_schema: Dict[str, str]
    health_endpoint: str
    ttl_seconds: int = 300            # Lease TTL — phải renew trước khi hết
    registered_at: float = field(default_factory=time.time)
    last_heartbeat: float = field(default_factory=time.time)
    status: str = "HEALTHY"           # HEALTHY | DEGRADED | UNHEALTHY

    def is_alive(self) -> bool:
        """Kiểm tra TTL lease còn hạn không"""
        return (time.time() - self.last_heartbeat) < self.ttl_seconds

    def renew_heartbeat(self):
        """Agent gửi heartbeat để renew lease"""
        self.last_heartbeat = time.time()

    def to_dict(self):
        return asdict(self)


# ─── Demo: In ra CapabilityCard
sample_card = CapabilityCard(
    agent_id="credit-agent-01",
    name="CreditAgent",
    capabilities=["credit_check", "cic_query", "score_analysis"],
    sla_ms=300,
    cost_per_call=0.002,
    input_schema={"loan_id": "str", "customer_id": "str"},
    output_schema={"credit_score": "int", "risk_grade": "str"},
    health_endpoint="/health/credit-agent-01"
)

print("🪪 CapabilityCard của CreditAgent:")
print(f"  Agent ID   : {sample_card.agent_id}")
print(f"  Capabilities: {sample_card.capabilities}")
print(f"  SLA        : {sample_card.sla_ms}ms")
print(f"  Cost/call  : ${sample_card.cost_per_call}")
print(f"  TTL        : {sample_card.ttl_seconds}s")
print(f"  Is alive?  : {sample_card.is_alive()}")

In [ ]:
# ─── Bước 1.2: Agent Registry — trung tâm discovery

class AgentRegistry:
    """DNS-inspired registry cho agents trong enterprise department."""

    def __init__(self):
        self._agents: Dict[str, CapabilityCard] = {}
        self._call_log: List[Dict] = []

    def register(self, card: CapabilityCard) -> str:
        self._agents[card.agent_id] = card
        print(f"  [Registry] ✅ Registered: {card.name} ({card.agent_id})")
        print(f"             Capabilities: {card.capabilities}")
        return card.agent_id

    def deregister(self, agent_id: str):
        if agent_id in self._agents:
            name = self._agents[agent_id].name
            del self._agents[agent_id]
            print(f"  [Registry] ❌ Deregistered: {name}")

    def heartbeat(self, agent_id: str):
        if agent_id in self._agents:
            self._agents[agent_id].renew_heartbeat()

    def find(self, capability: str, max_cost: float = 1.0) -> List[CapabilityCard]:
        """Tìm agents có capability, còn alive, trong budget.
        Returns: sorted by (health desc, cost asc)
        """
        # Cleanup expired leases
        expired = [aid for aid, c in self._agents.items() if not c.is_alive()]
        for aid in expired:
            print(f"  [Registry] ⚠️  Expired lease: {self._agents[aid].name}")
            del self._agents[aid]

        results = [
            c for c in self._agents.values()
            if capability in c.capabilities
            and c.cost_per_call <= max_cost
            and c.status == "HEALTHY"
        ]
        return sorted(results, key=lambda c: c.cost_per_call)

    def list_all(self):
        print(f"\n📋 Registry — {len(self._agents)} agents registered:")
        for aid, card in self._agents.items():
            alive = "🟢" if card.is_alive() else "🔴"
            print(f"  {alive} {card.name:20s} | {str(card.capabilities):45s} | SLA: {card.sla_ms}ms | ${card.cost_per_call}")


# ─── Khởi tạo registry global
registry = AgentRegistry()
print("🏗️  AgentRegistry khởi tạo xong")

In [ ]:
# ─── Bước 1.3: Đăng ký tất cả 8 agents vào registry
print("🚀 Đăng ký LoanBot v3.0 — Enterprise Department (8 agents)...\n")

AGENT_CARDS = [
    CapabilityCard(
        agent_id="credit-01", name="CreditAgent",
        capabilities=["credit_check", "cic_query", "score_analysis"],
        sla_ms=300, cost_per_call=0.002,
        input_schema={"loan_id": "str", "customer_id": "str"},
        output_schema={"credit_score": "int", "risk_grade": "str"},
        health_endpoint="/health/credit-01"
    ),
    CapabilityCard(
        agent_id="income-01", name="IncomeAgent",
        capabilities=["income_verify", "tax_check", "employment_verify"],
        sla_ms=400, cost_per_call=0.003,
        input_schema={"loan_id": "str", "customer_id": "str"},
        output_schema={"monthly_income": "float", "income_stable": "bool"},
        health_endpoint="/health/income-01"
    ),
    CapabilityCard(
        agent_id="fraud-01", name="FraudAgent",
        capabilities=["fraud_detection", "pattern_analysis", "blacklist_check"],
        sla_ms=250, cost_per_call=0.004,
        input_schema={"loan_id": "str", "customer_id": "str", "amount": "float"},
        output_schema={"fraud_score": "float", "fraud_flags": "list"},
        health_endpoint="/health/fraud-01"
    ),
    CapabilityCard(
        agent_id="risk-01", name="RiskAgent",
        capabilities=["risk_assessment", "dti_calculation", "collateral_eval"],
        sla_ms=500, cost_per_call=0.005,
        input_schema={"credit_score": "int", "monthly_income": "float", "loan_amount": "float"},
        output_schema={"risk_score": "float", "decision": "str", "interest_rate": "float"},
        health_endpoint="/health/risk-01"
    ),
    CapabilityCard(
        agent_id="compliance-01", name="ComplianceAgent",
        capabilities=["compliance_check", "hitl_gate", "ai_act_validation"],
        sla_ms=200, cost_per_call=0.001,
        input_schema={"loan_amount": "float", "decision": "str", "audit_trail": "dict"},
        output_schema={"compliant": "bool", "hitl_required": "bool", "violations": "list"},
        health_endpoint="/health/compliance-01"
    ),
    CapabilityCard(
        agent_id="report-01", name="ReportAgent",
        capabilities=["report_generation", "customer_notification", "summary_write"],
        sla_ms=600, cost_per_call=0.006,
        input_schema={"loan_id": "str", "all_results": "dict"},
        output_schema={"report_url": "str", "summary": "str"},
        health_endpoint="/health/report-01"
    ),
    CapabilityCard(
        agent_id="audit-01", name="AuditAgent",
        capabilities=["audit_logging", "immutable_record", "compliance_trail"],
        sla_ms=100, cost_per_call=0.0005,
        input_schema={"trace_id": "str", "all_events": "list"},
        output_schema={"audit_id": "str", "stored": "bool"},
        health_endpoint="/health/audit-01"
    ),
    CapabilityCard(
        agent_id="supervisor-01", name="IntelligentSupervisor",
        capabilities=["orchestration", "routing", "error_recovery"],
        sla_ms=2000, cost_per_call=0.01,
        input_schema={"loan_application": "dict"},
        output_schema={"final_decision": "dict"},
        health_endpoint="/health/supervisor-01"
    ),
]

for card in AGENT_CARDS:
    registry.register(card)

registry.list_all()

print("\n🔍 Test ANS Query: tìm agent có capability 'credit_check'")
found = registry.find("credit_check")
print(f"   → Tìm thấy: {[c.name for c in found]}")

---
## 🔷 Phần 2 — A2A Protocol Task State Machine

**Google Agent-to-Agent Protocol (April 2025):** Chuẩn hóa task handoffs giữa agents.

```
TaskState Machine:
  SUBMITTED → WORKING → COMPLETED
                      → INPUT_NEEDED (chờ thêm thông tin)
                      → FAILED
```

Mỗi task có: `task_id`, `state`, `artifacts` (kết quả không mất khi handoff), `history`.

In [ ]:
# ─── Bước 2.1: A2A TaskState Machine

class TaskState(str, Enum):
    SUBMITTED    = "SUBMITTED"
    WORKING      = "WORKING"
    INPUT_NEEDED = "INPUT_NEEDED"
    COMPLETED    = "COMPLETED"
    FAILED       = "FAILED"


@dataclass
class A2ATask:
    """A2A Protocol Task — standardized agent handoff object."""
    task_id: str
    capability_needed: str
    payload: Dict[str, Any]
    sender_id: str
    state: TaskState = TaskState.SUBMITTED
    artifacts: Dict[str, Any] = field(default_factory=dict)  # Kết quả, không mất khi handoff
    history: List[Dict] = field(default_factory=list)
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    updated_at: str = field(default_factory=lambda: datetime.now().isoformat())
    error: Optional[str] = None

    def transition(self, new_state: TaskState, note: str = "", result: Dict = None):
        old_state = self.state
        self.state = new_state
        self.updated_at = datetime.now().isoformat()
        event = {
            "from": old_state.value,
            "to": new_state.value,
            "note": note,
            "timestamp": self.updated_at
        }
        self.history.append(event)
        if result:
            self.artifacts.update(result)
        print(f"  [A2A] Task {self.task_id}: {old_state.value} → {new_state.value}" +
              (f" | {note}" if note else ""))

    def is_done(self) -> bool:
        return self.state in (TaskState.COMPLETED, TaskState.FAILED)


# ─── Demo: Lifecycle của một A2A Task
print("🔄 Demo A2A Task Lifecycle:")
demo_task = A2ATask(
    task_id="T-2024-001",
    capability_needed="credit_check",
    payload={"customer_id": "FC-2024-001", "loan_amount": 150_000_000},
    sender_id="supervisor-01"
)
print(f"  Created: state={demo_task.state.value}")
demo_task.transition(TaskState.WORKING, "CreditAgent started")
demo_task.transition(TaskState.COMPLETED, "Score retrieved", {"credit_score": 720, "risk_grade": "B"})
print(f"\n  Artifacts: {demo_task.artifacts}")
print(f"  History:   {len(demo_task.history)} events")

---
## 🔷 Phần 3 — State Checkpointing & Resume

**Problem:** Server crash giữa chừng → mất toàn bộ kết quả đã tính.

**Solution:** Checkpoint state sau mỗi agent call → khi restart, load checkpoint và tiếp tục từ bước đã hoàn thành.

In [ ]:
# ─── Bước 3.1: CheckpointManager
# Production: lưu vào PostgreSQL/Redis. Lab: lưu vào dict (in-memory)

class CheckpointManager:
    """Persist & restore pipeline state. In production: PostgreSQL + Redis."""

    def __init__(self):
        self._store: Dict[str, Dict] = {}  # loan_id -> checkpoint

    def save(self, loan_id: str, step: int, completed_agents: List[str], results: Dict):
        checkpoint = {
            "loan_id": loan_id,
            "step": step,
            "completed_agents": completed_agents,
            "results": results,
            "saved_at": datetime.now().isoformat()
        }
        self._store[loan_id] = checkpoint
        # Hash để detect tampering
        payload = json.dumps(checkpoint, sort_keys=True)
        checkpoint["checksum"] = hashlib.sha256(payload.encode()).hexdigest()[:12]
        print(f"  [Checkpoint] 💾 Saved step {step} for {loan_id} | checksum: {checkpoint['checksum']}")
        return checkpoint

    def load(self, loan_id: str) -> Optional[Dict]:
        cp = self._store.get(loan_id)
        if cp:
            print(f"  [Checkpoint] 📂 Loaded step {cp['step']} for {loan_id}")
            print(f"               Completed agents: {cp['completed_agents']}")
        return cp

    def delete(self, loan_id: str):
        if loan_id in self._store:
            del self._store[loan_id]


# ─── Demo: Checkpoint & Resume
cp_mgr = CheckpointManager()

print("💾 Demo Checkpoint & Resume:")
print("\n  Bước 1: Pipeline chạy Phase 1 (CreditAgent + IncomeAgent + FraudAgent)")
cp_mgr.save(
    loan_id="FC-2024-001",
    step=1,
    completed_agents=["CreditAgent", "IncomeAgent", "FraudAgent"],
    results={
        "credit_score": 720,
        "monthly_income": 25_000_000,
        "fraud_score": 0.12
    }
)

print("\n  ⚡ SERVER CRASH! Khởi động lại...")
print("\n  Bước 2: Load checkpoint, resume từ Phase 2")
restored = cp_mgr.load("FC-2024-001")
print(f"  Kết quả đã có: {list(restored['results'].keys())}")
print("  ✅ Không cần gọi lại CIC API — tiết kiệm tiền và thời gian!")

---
## 🔷 Phần 4 — Circuit Breaker cho External APIs

Nhắc lại từ Module 2 — áp dụng vào context multi-agent để ngăn cascading failures.

In [ ]:
# ─── Bước 4.1: Circuit Breaker — tái sử dụng từ Module 2

class CircuitState(str, Enum):
    CLOSED    = "CLOSED"    # Hoạt động bình thường
    OPEN      = "OPEN"      # Đang chặn — quá nhiều lỗi
    HALF_OPEN = "HALF_OPEN" # Đang thử lại


class CircuitBreaker:
    def __init__(self, name: str, failure_threshold=3, timeout_seconds=30):
        self.name = name
        self.failure_threshold = failure_threshold
        self.timeout_seconds = timeout_seconds
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.last_failure_time = None

    def call(self, func, *args, fallback=None, **kwargs):
        if self.state == CircuitState.OPEN:
            elapsed = time.time() - self.last_failure_time
            if elapsed > self.timeout_seconds:
                self.state = CircuitState.HALF_OPEN
                print(f"  [CB:{self.name}] OPEN → HALF_OPEN (testing...)")
            else:
                print(f"  [CB:{self.name}] ⚡ OPEN — returning fallback")
                return fallback
        try:
            result = func(*args, **kwargs)
            if self.state == CircuitState.HALF_OPEN:
                self.state = CircuitState.CLOSED
                self.failure_count = 0
                print(f"  [CB:{self.name}] HALF_OPEN → CLOSED ✅")
            return result
        except Exception as e:
            self.failure_count += 1
            self.last_failure_time = time.time()
            if self.failure_count >= self.failure_threshold:
                self.state = CircuitState.OPEN
                print(f"  [CB:{self.name}] → OPEN ❌ (failures: {self.failure_count})")
            raise e


# Test Circuit Breaker
cic_breaker = CircuitBreaker("CIC_API", failure_threshold=3)

def mock_cic_api_call(should_fail=False):
    if should_fail:
        raise ConnectionError("CIC API timeout")
    return {"credit_score": 720}

print("🔌 Test Circuit Breaker với CIC API:")
print("  Call 1 (success):", cic_breaker.call(mock_cic_api_call, fallback={"credit_score": 500}))
for i in range(3):
    try:
        cic_breaker.call(mock_cic_api_call, should_fail=True, fallback={"credit_score": 500})
    except:
        print(f"  Call {i+2} (fail): Exception caught")
print("  Call 5 (breaker OPEN):", cic_breaker.call(mock_cic_api_call, fallback={"credit_score": 500}))

---
## 🔷 Phần 5 — LoanBot v3.0: 8 Specialized Agents (Claude API)

Tổng hợp tất cả: Registry + A2A + Checkpoint + CircuitBreaker + 8 agents thật.

```
Kiến trúc 3 Phase:
  Phase 1 (Parallel):  CreditAgent ─┐
                       IncomeAgent  ─┼─→ Phase 2
                       FraudAgent  ─┘

  Phase 2 (Parallel):  RiskAgent ────┐
                       ComplianceAgent─┼─→ Phase 3

  Phase 3 (Parallel):  ReportAgent ──┐
                       AuditAgent ───┘
```

In [ ]:
# ─── Bước 5.1: BaseAgent — template cho tất cả agents

class BaseAgent:
    def __init__(self, agent_id: str, name: str, capability: str):
        self.agent_id = agent_id
        self.name = name
        self.capability = capability
        self._breaker = CircuitBreaker(name, failure_threshold=3)

    def _call_claude(self, system_prompt: str, user_message: str) -> str:
        """Gọi Claude API và trả về text response."""
        response = client.messages.create(
            model=MODEL,
            max_tokens=400,
            messages=[{"role": "user", "content": user_message}],
            system=system_prompt
        )
        return response.content[0].text

    def _parse_json(self, text: str) -> Dict:
        """Trích xuất JSON từ response text."""
        try:
            start = text.find('{')
            end = text.rfind('}') + 1
            if start >= 0 and end > start:
                return json.loads(text[start:end])
        except:
            pass
        return {}

    async def process(self, task: A2ATask) -> A2ATask:
        raise NotImplementedError

print("✅ BaseAgent template defined")

In [ ]:
# ─── Bước 5.2: 8 Specialized Agents

class CreditAgent(BaseAgent):
    def __init__(self):
        super().__init__("credit-01", "CreditAgent", "credit_check")

    async def process(self, task: A2ATask) -> A2ATask:
        task.transition(TaskState.WORKING, "Querying CIC database")
        payload = task.payload
        prompt = f"""Bạn là CreditAgent của LoanBot. Phân tích hồ sơ tín dụng.
Customer ID: {payload.get('customer_id')}
Loan amount: {payload.get('loan_amount', 0):,.0f} VNĐ
Credit history: {payload.get('credit_history', 'No defaults in 3 years')}
Trả về JSON: {{"credit_score": int(500-850), "risk_grade": str("A"/"B"/"C"/"D"), "cic_remarks": str}}"""
        raw = self._call_claude("You are a credit scoring specialist. Always respond with valid JSON only.", prompt)
        result = self._parse_json(raw)
        if not result:
            result = {"credit_score": 700, "risk_grade": "B", "cic_remarks": "No negative records"}
        task.transition(TaskState.COMPLETED, "Credit score computed", result)
        return task


class IncomeAgent(BaseAgent):
    def __init__(self):
        super().__init__("income-01", "IncomeAgent", "income_verify")

    async def process(self, task: A2ATask) -> A2ATask:
        task.transition(TaskState.WORKING, "Verifying income via tax records")
        payload = task.payload
        prompt = f"""Bạn là IncomeAgent. Xác minh thu nhập của người vay.
Customer ID: {payload.get('customer_id')}
Declared income: {payload.get('declared_income', 20000000):,.0f} VNĐ/tháng
Employment: {payload.get('employment', 'Full-time, 5 years')}
Trả về JSON: {{"monthly_income": float, "income_verified": bool, "income_stability": str("HIGH"/"MEDIUM"/"LOW")}}"""
        raw = self._call_claude("You are an income verification specialist. Always respond with valid JSON only.", prompt)
        result = self._parse_json(raw)
        if not result:
            result = {"monthly_income": 25000000, "income_verified": True, "income_stability": "HIGH"}
        task.transition(TaskState.COMPLETED, "Income verified", result)
        return task


class FraudAgent(BaseAgent):
    def __init__(self):
        super().__init__("fraud-01", "FraudAgent", "fraud_detection")

    async def process(self, task: A2ATask) -> A2ATask:
        task.transition(TaskState.WORKING, "Running fraud detection patterns")
        payload = task.payload
        prompt = f"""Bạn là FraudAgent. Phân tích rủi ro gian lận.
Customer ID: {payload.get('customer_id')}
Loan amount: {payload.get('loan_amount', 0):,.0f} VNĐ
Application patterns: {payload.get('patterns', 'First-time applicant, normal behavior')}
Trả về JSON: {{"fraud_score": float(0.0-1.0), "fraud_flags": list[str], "recommendation": str("PROCEED"/"FLAG"/"REJECT")}}"""
        raw = self._call_claude("You are a fraud detection specialist. Always respond with valid JSON only.", prompt)
        result = self._parse_json(raw)
        if not result:
            result = {"fraud_score": 0.08, "fraud_flags": [], "recommendation": "PROCEED"}
        task.transition(TaskState.COMPLETED, "Fraud analysis complete", result)
        return task


class RiskAgent(BaseAgent):
    def __init__(self):
        super().__init__("risk-01", "RiskAgent", "risk_assessment")

    async def process(self, task: A2ATask) -> A2ATask:
        task.transition(TaskState.WORKING, "Computing risk score & decision")
        payload = task.payload
        loan_amount = payload.get('loan_amount', 0)
        monthly_income = payload.get('monthly_income', 1)
        dti = (loan_amount / 12) / monthly_income if monthly_income > 0 else 1
        prompt = f"""Bạn là RiskAgent. Đánh giá rủi ro tổng thể và ra quyết định.
Credit score: {payload.get('credit_score', 650)}
Monthly income: {monthly_income:,.0f} VNĐ
Loan amount: {loan_amount:,.0f} VNĐ
DTI ratio: {dti:.2f}
Fraud score: {payload.get('fraud_score', 0.1)}
Trả về JSON: {{"risk_score": float(0-100), "decision": str("APPROVED"/"REJECTED"/"MANUAL_REVIEW"), "interest_rate": float, "reasoning": str}}"""
        raw = self._call_claude("You are a loan risk assessment specialist. Always respond with valid JSON only.", prompt)
        result = self._parse_json(raw)
        if not result:
            result = {"risk_score": 35.0, "decision": "APPROVED", "interest_rate": 8.5, "reasoning": "Good credit profile"}
        task.transition(TaskState.COMPLETED, "Risk decision made", result)
        return task


class ComplianceAgent(BaseAgent):
    def __init__(self):
        super().__init__("compliance-01", "ComplianceAgent", "compliance_check")
        self.HITL_THRESHOLD = 500_000_000  # 500 triệu VNĐ

    async def process(self, task: A2ATask) -> A2ATask:
        task.transition(TaskState.WORKING, "Checking EU AI Act compliance")
        payload = task.payload
        loan_amount = payload.get('loan_amount', 0)
        hitl_required = loan_amount > self.HITL_THRESHOLD
        violations = []
        if hitl_required and payload.get('decision') == 'APPROVED' and not payload.get('human_reviewed'):
            violations.append("Art.14 HITL violation: loan >500M requires human review")
        result = {
            "compliant": len(violations) == 0,
            "hitl_required": hitl_required,
            "violations": violations,
            "ai_act_tier": "HIGH_RISK" if hitl_required else "LIMITED_RISK",
            "art12_logged": True
        }
        task.transition(TaskState.COMPLETED, "Compliance check done", result)
        return task


class ReportAgent(BaseAgent):
    def __init__(self):
        super().__init__("report-01", "ReportAgent", "report_generation")

    async def process(self, task: A2ATask) -> A2ATask:
        task.transition(TaskState.WORKING, "Generating loan decision report")
        payload = task.payload
        prompt = f"""Bạn là ReportAgent. Viết tóm tắt quyết định vay bằng tiếng Việt (3-4 câu).
Hồ sơ: {payload.get('loan_id')}
Quyết định: {payload.get('decision', 'APPROVED')}
Credit score: {payload.get('credit_score', 700)}
Lãi suất: {payload.get('interest_rate', 8.5)}%/năm
Compliant: {payload.get('compliant', True)}
HITL required: {payload.get('hitl_required', False)}"""
        summary = self._call_claude("Bạn là chuyên gia viết báo cáo tín dụng ngân hàng. Viết ngắn gọn, chuyên nghiệp.", prompt)
        result = {
            "report_url": f"/reports/{payload.get('loan_id', 'unknown')}.pdf",
            "summary": summary.strip()
        }
        task.transition(TaskState.COMPLETED, "Report generated", result)
        return task


class AuditAgent(BaseAgent):
    def __init__(self):
        super().__init__("audit-01", "AuditAgent", "audit_logging")
        self._audit_log: List[Dict] = []

    async def process(self, task: A2ATask) -> A2ATask:
        task.transition(TaskState.WORKING, "Writing immutable audit trail")
        audit_record = {
            "audit_id": f"AUDIT-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
            "loan_id": task.payload.get('loan_id'),
            "events": task.payload.get('all_events', []),
            "final_decision": task.payload.get('decision'),
            "timestamp": datetime.now().isoformat(),
            "retention": "3 years (EU AI Act Art.12)",
            "immutable": True
        }
        self._audit_log.append(audit_record)
        result = {"audit_id": audit_record["audit_id"], "stored": True, "retention": audit_record["retention"]}
        task.transition(TaskState.COMPLETED, "Audit trail persisted", result)
        return task


print("✅ 7 specialized agents defined (CreditAgent, IncomeAgent, FraudAgent, RiskAgent, ComplianceAgent, ReportAgent, AuditAgent)")

In [ ]:
# ─── Bước 5.3: IntelligentSupervisor — orchestrates all 8 agents

class IntelligentSupervisor:
    """Orchestrates 8-agent enterprise department với Registry + A2A + Checkpoint."""

    def __init__(self, reg: AgentRegistry, cp: CheckpointManager):
        self.registry = reg
        self.checkpoint = cp
        self.agents = {
            "credit_check":    CreditAgent(),
            "income_verify":   IncomeAgent(),
            "fraud_detection": FraudAgent(),
            "risk_assessment": RiskAgent(),
            "compliance_check":ComplianceAgent(),
            "report_generation":ReportAgent(),
            "audit_logging":   AuditAgent(),
        }
        self.all_events: List[str] = []

    def _log(self, msg: str):
        ts = datetime.now().strftime("%H:%M:%S.%f")[:12]
        line = f"[{ts}] {msg}"
        self.all_events.append(line)
        print(line)

    async def _run_agent(self, capability: str, payload: Dict, sender: str) -> Dict:
        """Tìm agent qua Registry, tạo A2A Task, chạy và trả về artifacts."""
        # ANS Query
        candidates = self.registry.find(capability)
        if not candidates:
            raise RuntimeError(f"No healthy agent for capability: {capability}")
        agent_card = candidates[0]
        self._log(f"→ ANS: '{capability}' resolved to {agent_card.name}")

        # Create A2A Task
        task = A2ATask(
            task_id=f"{capability[:6].upper()}-{datetime.now().strftime('%H%M%S')}",
            capability_needed=capability,
            payload=payload,
            sender_id=sender
        )

        # Run agent
        agent = self.agents[capability]
        completed_task = await agent.process(task)
        return completed_task.artifacts

    async def process_loan(self, application: Dict) -> Dict:
        loan_id = application['loan_id']
        loan_amount = application['loan_amount']
        trace_id = f"TRACE-{loan_id}-{datetime.now().strftime('%Y%m%d%H%M%S')}"

        self._log(f"🏢 === LoanBot v3.0 Enterprise Department ===")
        self._log(f"📋 Hồ sơ: {loan_id} | Khoản vay: {loan_amount:,.0f} VNĐ | Trace: {trace_id}")

        # Check for existing checkpoint (Resume)
        checkpoint = self.checkpoint.load(loan_id)
        results = checkpoint['results'].copy() if checkpoint else {}
        resume_from = checkpoint['step'] + 1 if checkpoint else 0
        if checkpoint:
            self._log(f"📂 Resuming từ checkpoint step {resume_from}")

        base_payload = {**application, **results}

        # ── Phase 1: Parallel — Credit + Income + Fraud
        if resume_from <= 1:
            self._log("\n⚡ Phase 1: Parallel (CreditAgent + IncomeAgent + FraudAgent)")
            t0 = time.time()
            p1_results = await asyncio.gather(
                self._run_agent("credit_check",    base_payload, "supervisor-01"),
                self._run_agent("income_verify",   base_payload, "supervisor-01"),
                self._run_agent("fraud_detection", base_payload, "supervisor-01"),
            )
            results.update(p1_results[0])
            results.update(p1_results[1])
            results.update(p1_results[2])
            elapsed = time.time() - t0
            self._log(f"✅ Phase 1 done in {elapsed:.2f}s")
            # Checkpoint sau Phase 1
            self.checkpoint.save(loan_id, 1, ["CreditAgent", "IncomeAgent", "FraudAgent"], results)

        # ── Phase 2: Parallel — Risk + Compliance
        if resume_from <= 2:
            self._log("\n⚡ Phase 2: Parallel (RiskAgent + ComplianceAgent)")
            t0 = time.time()
            phase2_payload = {**base_payload, **results}
            p2_results = await asyncio.gather(
                self._run_agent("risk_assessment",  phase2_payload, "supervisor-01"),
                self._run_agent("compliance_check", phase2_payload, "supervisor-01"),
            )
            results.update(p2_results[0])
            results.update(p2_results[1])
            elapsed = time.time() - t0
            self._log(f"✅ Phase 2 done in {elapsed:.2f}s")
            # Checkpoint sau Phase 2
            self.checkpoint.save(loan_id, 2, ["CreditAgent", "IncomeAgent", "FraudAgent", "RiskAgent", "ComplianceAgent"], results)

            # HITL Gate
            if results.get('hitl_required') and not application.get('human_reviewed'):
                self._log(f"⚠️  HITL GATE: Loan >500M VNĐ → Yêu cầu reviewer con người (Art.14)")
                return {"status": "PENDING_HUMAN_REVIEW", "loan_id": loan_id, "results": results}

        # ── Phase 3: Parallel — Report + Audit
        self._log("\n⚡ Phase 3: Parallel (ReportAgent + AuditAgent)")
        t0 = time.time()
        phase3_payload = {**base_payload, **results, "all_events": self.all_events}
        p3_results = await asyncio.gather(
            self._run_agent("report_generation", phase3_payload, "supervisor-01"),
            self._run_agent("audit_logging",     phase3_payload, "supervisor-01"),
        )
        results.update(p3_results[0])
        results.update(p3_results[1])
        elapsed = time.time() - t0
        self._log(f"✅ Phase 3 done in {elapsed:.2f}s")

        # Cleanup checkpoint — pipeline hoàn thành
        self.checkpoint.delete(loan_id)

        return {
            "status": "COMPLETED",
            "loan_id": loan_id,
            "trace_id": trace_id,
            "decision": results.get('decision', 'N/A'),
            "credit_score": results.get('credit_score', 0),
            "interest_rate": results.get('interest_rate', 0),
            "compliant": results.get('compliant', False),
            "hitl_required": results.get('hitl_required', False),
            "report_url": results.get('report_url', ''),
            "audit_id": results.get('audit_id', ''),
            "summary": results.get('summary', ''),
            "all_results": results
        }


print("✅ IntelligentSupervisor defined — 3-phase orchestration với Registry + A2A + Checkpoint")

In [ ]:
# ─── Bước 5.4: Chạy LoanBot v3.0 — Case 1: Hồ sơ tốt (FC-2024-001)
# Lê Văn A | Thu nhập 25M/tháng | Vay 150M — dưới ngưỡng HITL

print("="*70)
print("🧾 CASE 1: FC-2024-001 — Lê Văn A (hồ sơ tốt, dưới ngưỡng HITL)")
print("="*70)

supervisor = IntelligentSupervisor(registry, cp_mgr)

application_1 = {
    "loan_id": "FC-2024-001",
    "customer_id": "CUST-001",
    "customer_name": "Lê Văn A",
    "loan_amount": 150_000_000,       # 150 triệu VNĐ — dưới 500M HITL
    "declared_income": 25_000_000,    # 25 triệu/tháng
    "credit_history": "No defaults in 5 years, excellent payment record",
    "employment": "Senior engineer, 7 years at tech company",
    "patterns": "Regular applicant, consistent financial behavior"
}

result_1 = await supervisor.process_loan(application_1)

print("\n" + "─"*70)
print("📊 KẾT QUẢ CUỐI CÙNG:")
print(f"  ✅ Status      : {result_1['status']}")
print(f"  📋 Decision    : {result_1['decision']}")
print(f"  📈 Credit Score: {result_1['credit_score']}")
print(f"  💰 Interest Rate: {result_1['interest_rate']}%/năm")
print(f"  ⚖️  Compliant  : {result_1['compliant']}")
print(f"  👤 HITL Req'd : {result_1['hitl_required']}")
print(f"  📄 Report URL : {result_1['report_url']}")
print(f"  🔒 Audit ID   : {result_1['audit_id']}")
print(f"\n  📝 SUMMARY:\n{result_1['summary']}")

In [ ]:
# ─── Case 2: Hồ sơ lớn (FC-2024-002) — Vượt ngưỡng HITL 500M
print("="*70)
print("🧾 CASE 2: FC-2024-002 — Trần Thị B (vay 800M — VỢT ngưỡng HITL)")
print("="*70)

supervisor2 = IntelligentSupervisor(registry, cp_mgr)

application_2 = {
    "loan_id": "FC-2024-002",
    "customer_id": "CUST-002",
    "customer_name": "Trần Thị B",
    "loan_amount": 800_000_000,        # 800 triệu — VƯỢT ngưỡng HITL 500M
    "declared_income": 60_000_000,     # 60 triệu/tháng
    "credit_history": "Good payment history, 1 late payment 2 years ago",
    "employment": "Business owner, 10 years",
    "patterns": "First large loan application",
    "human_reviewed": False            # Chưa có human review
}

result_2 = await supervisor2.process_loan(application_2)

print("\n" + "─"*70)
print("📊 KẾT QUẢ CUỐI CÙNG:")
print(f"  Status    : {result_2['status']}")
print(f"  HITL Gate : {'🛑 BLOCKED — Cần human reviewer' if result_2['status'] == 'PENDING_HUMAN_REVIEW' else '✅ Passed'}")
if 'decision' in result_2:
    print(f"  Decision  : {result_2['decision']}")

---
## 🏋️ Bài Tập Thực Hành

### Bài 1 — Thêm MarketingAgent vào Registry
Tạo `MarketingAgent` với capability `send_offer`. Đăng ký vào registry và gọi từ Supervisor sau khi loan được APPROVED.

### Bài 2 — Simulate Server Crash
Chạy pipeline, trong code thêm `raise Exception("Crash!")` sau Phase 1. Restart và verify checkpoint resume hoạt động đúng.

### Bài 3 — Circuit Breaker Test
Modify `CreditAgent._call_claude()` để throw exception 3 lần. Verify Circuit Breaker mở mạch và trả về fallback score.

In [ ]:
# ─── Bài 1: Thêm MarketingAgent
# TODO: Viết code của bạn ở đây

class MarketingAgent(BaseAgent):
    def __init__(self):
        super().__init__("marketing-01", "MarketingAgent", "send_offer")

    async def process(self, task: A2ATask) -> A2ATask:
        # TODO: Implement marketing offer logic
        # Gợi ý: Dùng self._call_claude() để generate personalized offer
        pass


# TODO: Đăng ký vào registry
# marketing_card = CapabilityCard(
#     agent_id="marketing-01",
#     name="MarketingAgent",
#     capabilities=["send_offer", "cross_sell"],
#     ...
# )
# registry.register(marketing_card)

print("💡 Hoàn thiện MarketingAgent và đăng ký vào registry!")

In [ ]:
# ─── Bài 2: Simulate & Test Checkpoint Resume
# TODO: Implement crash simulation

class FaultInjectSupervisor(IntelligentSupervisor):
    """Supervisor với fault injection để test checkpoint/resume."""

    def __init__(self, *args, crash_after_phase=1, **kwargs):
        super().__init__(*args, **kwargs)
        self.crash_after_phase = crash_after_phase

    async def process_loan(self, application: Dict) -> Dict:
        # TODO: Override process_loan để inject crash sau phase được chỉ định
        # Gợi ý: Sau khi Phase 1 hoàn thành và checkpoint saved,
        # raise Exception("⚡ SIMULATED CRASH!")
        pass

print("💡 Implement FaultInjectSupervisor và test crash/resume flow!")

# Khi xong, test như sau:
# try:
#     fault_supervisor = FaultInjectSupervisor(registry, cp_mgr, crash_after_phase=1)
#     await fault_supervisor.process_loan(application_1)
# except Exception as e:
#     print(f"Crashed: {e}")
#     print("Restarting...")
#     recovery_supervisor = IntelligentSupervisor(registry, cp_mgr)
#     result = await recovery_supervisor.process_loan(application_1)  # Should resume from checkpoint
#     print(f"Recovered: {result['decision']}")

---
## 📊 Tổng Kết Module 7

Bạn đã xây dựng thành công **LoanBot v3.0 Enterprise Department**:

| Component | Implemented | Concept |
|-----------|------------|--------|
| Agent Registry + ANS | ✅ | Capability-based discovery, TTL lease |
| Capability Card | ✅ | Agent CV: SLA, cost, schema, health |
| A2A Task State Machine | ✅ | SUBMITTED→WORKING→COMPLETED |
| Checkpoint & Resume | ✅ | Persist state, recover from crash |
| Circuit Breaker | ✅ | Prevent cascading failures |
| 8 Specialized Agents | ✅ | Credit + Income + Fraud + Risk + Compliance + Report + Audit + Supervisor |
| Parallel Execution | ✅ | asyncio.gather, 3-phase architecture |
| HITL Gate | ✅ | EU AI Act Art.14, >500M threshold |
| Audit Logging | ✅ | Immutable, 3-year retention (Art.12) |

### 🎯 Key Takeaways
- **23% success rate** của enterprise multi-agent scale → 3 vấn đề chính: hardcoded architecture, state management kém, cascading failures
- **Registry Pattern** giải quyết hardcoding: Supervisor query theo capability, không theo tên
- **Checkpoint** giải quyết state loss: persist trước mỗi external call, resume sau crash
- **Circuit Breaker** giải quyết cascading: fail fast, fallback, self-heal

---
**Chương trình đào tạo AI Agent Harness Engineering — FinTech Corp 2024**

*Module 7/7 hoàn thành* · *LoanBot v3.0 — Production Ready*